In [1]:
import pandas as pd
import os
from pathlib import Path
import duckdb
from collections import defaultdict
import string

In [2]:
CWD_DIR = os.getcwd()
RAW_DIR = os.path.join(CWD_DIR, "data\\raw")
USAGE_DIR = os.path.join(RAW_DIR, "usage-stats")

In [3]:
def read_csv_header(file_name):
    try:
        return pd.read_csv(file_name, low_memory=False, nrows = 0).columns.tolist()
    except UnicodeDecodeError:
        return pd.read_csv(file_name, low_memory=False, nrows=0, encoding="latin-1")


In [4]:
def read_csv_durable(file_name):
    try:
        return pd.read_csv(file_name, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(file_name, low_memory=False, encoding="latin-1")


In [5]:
target_dir = Path(USAGE_DIR)

csv_names = [file.name for file in target_dir.iterdir() if file.suffix == ".csv"]

In [6]:
schema_map: dict[frozenset, list[str]] = defaultdict(list)

In [7]:
for file_name in csv_names:
    cols = read_csv_header(os.path.join(USAGE_DIR, file_name))
    schema_map[frozenset(cols)].append(file_name)

In [8]:
lengths = {key: len(value) for key, value in schema_map.items()}

In [9]:
SCHEMAS = [
    # v1_standard : 403 + 2 + 12 files; The 2 + 12 cases are due to unnamed columns
    {
        "Rental Id": "rental_id", "Duration": "duration_seconds", "Bike Id": "bike_id",
        "End Date": "end_date", "EndStation Id": "end_station_id", "EndStation Name": "end_station_name",
        "Start Date": "start_date", "StartStation Id": "start_station_id", "StartStation Name": "start_station_name",
    },
    # v1_logical : v1 but with logical terminals
    {
        "Rental Id": "rental_id", "Duration": "duration_seconds", "Bike Id": "bike_id",
        "End Date": "end_date", "EndStation Logical Terminal": "end_station_id", "EndStation Name": "end_station_name",
        "Start Date": "start_date", "StartStation Logical Terminal": "start_station_id", "StartStation Name": "start_station_name",
    },
    # v1_no_endstation : v1 but missing end station
    {
        "Rental Id": "rental_id", "Duration": "duration_seconds", "Bike Id": "bike_id",
        "End Date": "end_date", "EndStation Name": "end_station_name",
        "Start Date": "start_date", "StartStation Id": "start_station_id", "StartStation Name": "start_station_name",
    },
    # v2 : has spaced station names and Duration_Seconds
    {
        "Rental Id": "rental_id", "Duration_Seconds": "duration_seconds", "Bike Id": "bike_id",
        "End Date": "end_date", "End Station Id": "end_station_id", "End Station Name": "end_station_name",
        "Start Date": "start_date", "Start Station Id": "start_station_id", "Start Station Name": "start_station_name",
    },
    # v3 : identified by Number and Total duration (ms); Total duration ignored
    {
        "Number": "rental_id", "Bike number": "bike_id", "Bike model": "bike_model",
        "Start date": "start_date", "End date": "end_date",
        "Start station number": "start_station_id", "Start station": "start_station_name",
        "End station number": "end_station_id", "End station": "end_station_name",
        "Total duration (ms)": "duration_ms",
    },
]

In [10]:
OUTPUT_COLS = [
    "rental_id","bike_id","bike_model",
    "start_date","end_date",
    "start_station_id","start_station_name",
    "end_station_id","end_station_name",
    "duration_seconds",
]

In [11]:
#lets normalise this data


def normalise_df(df) -> pd.DataFrame:
    #identify schema
    col_set = set(df.columns)
    #because of the cases with extra leftover cols on the data, we want to use a subse
    for s in SCHEMAS:
        if set(s).issubset(col_set):
            schema = s
            break
    if schema is None:
        raise ValueError("Unrecognised Schema :(")

    cols_to_drop = [c for c in df.columns if c.startswith("Unnamed") or c == "Total Duration"]
    df = df.drop(columns = cols_to_drop)
    df = df.rename(columns=schema)[list(schema.values())]

    if "duration_ms" in df.columns:
        df["duration_seconds"] = (pd.to_numeric(df.pop("duration_ms") / 100)).round()
    
    for col in OUTPUT_COLS:
        if col not in df.columns:
            df[col] = pd.NA

    DATE_FORMAT = "%d/%m/%Y %H:%M"

    df["start_date"] = pd.to_datetime(df["start_date"], format=DATE_FORMAT, errors="coerce")
    df["end_date"]   = pd.to_datetime(df["end_date"],   format=DATE_FORMAT, errors="coerce")

    for col in ("rental_id","bike_id","start_station_id","end_station_id","duration_seconds"):
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df[OUTPUT_COLS]
    


In [12]:
test_df = pd.read_csv(os.path.join(USAGE_DIR, csv_names[370]))
test_df

,Number,Start date,Start station number,Start station,End date,End station number,End station,Bike number,Bike model,Total duration,Total duration (ms)
0,133176609,2023-08-14 23:59,1028,"William IV Street, Strand",2023-08-15 00:17,1050,"Park Road (Baker Street), The Regent's Park",56448,CLASSIC,17m 54s,1074273
1,133176610,2023-08-14 23:59,3430,"Central House, Aldgate",2023-08-15 00:12,1219,"Lower Marsh, Waterloo",53250,CLASSIC,13m 23s,803995
2,133176611,2023-08-14 23:59,200251,"Bow Road Station, Bow",2023-08-15 00:04,200006,"Hewison Street, Old Ford",23215,CLASSIC,4m 13s,253588
3,133176605,2023-08-14 23:58,2703,"Fire Brigade Pier, Vauxhall",2023-08-15 00:07,200084,"Doddington Grove, Kennington",42167,CLASSIC,9m 37s,577273
4,133176607,2023-08-14 23:58,1153,"Pall Mall East, West End",2023-08-15 00:37,3452,"Panton Street, West End",51973,CLASSIC,38m 51s,2331917
...,...,...,...,...,...,...,...,...,...,...,...
341371,132825197,2023-08-01 00:01,300202,"Kings Gate House, Westminster",2023-08-01 00:18,1210,"Nevern Place, Earl's Court",51757,CLASSIC,17m 5s,1025908
341372,132825198,2023-08-01 00:01,1154,"Kennington Road , Vauxhall",2023-08-01 00:09,1093,"Kennington Cross, Kennington",53424,CLASSIC,7m 16s,436692
341373,132825189,2023-08-01 00:00,1190,"Kennington Lane Rail Bridge, Vauxhall",2023-08-01 00:17,1059,"Albert Embankment, Vauxhall",23715,CLASSIC,16m 46s,1006663
341374,132825190,2023-08-01 00:00,1190,"Kennington Lane Rail Bridge, Vauxhall",2023-08-01 00:17,1059,"Albert Embankment, Vauxhall",41267,CLASSIC,16m 47s,1007128


In [13]:
usage_db = duckdb.connect("usage_alt_db.duckdb")

In [14]:
usage_db.execute(f"""
    CREATE TABLE IF NOT EXISTS trips (
        rental_id        BIGINT,
        bike_id          BIGINT,
        bike_model       VARCHAR,
        start_date       TIMESTAMP,
        end_date         TIMESTAMP,
        start_station_id BIGINT,
        start_station_name VARCHAR,
        end_station_id   BIGINT,
        end_station_name VARCHAR,
        duration_seconds DOUBLE
    )
""")

In [ ]:
iter = 0
CHECKPOINT_EVERY = 10
for file_name in csv_names:
    iter += 1
    print(iter)
    print(file_name)
    chunk = normalise_df(read_csv_durable(os.path.join(USAGE_DIR, file_name)))
    chunk = chunk.dropna(subset=["rental_id","start_date","end_date"])
    chunk = chunk[OUTPUT_COLS]
    usage_db.execute("INSERT INTO trips SELECT * FROM chunk")
    del chunk

    if iter % CHECKPOINT_EVERY == 0:
        usage_db.execute("CHECKPOINT")
        print(f"checkpoint at file {iter}")
    

1
01aJourneyDataExtract10Jan16-23Jan16.csv
2
01b Journey Data Extract 24Jan16-06Feb16.csv
3
01bJourneyDataExtract24Jan16-06Feb16.csv
4
02aJourneyDataExtract07Fe16-20Feb2016.csv
5
02aJourneyDataExtract07Feb16-20Feb2016.csv
6
02bJourneyDataExtract21Feb16-05Mar2016.csv
7
03JourneyDataExtract06Mar2016-31Mar2016.csv
8
04JourneyDataExtract01Apr2016-30Apr2016.csv
9
05JourneyDataExtract01May2016-17May2016.csv
10
06JourneyDataExtract18May2016-24May2016.csv
11
07JourneyDataExtract25May2016-31May2016.csv
12
08JourneyDataExtract01Jun2016-07Jun2016.csv
13
09JourneyDataExtract08Jun2016-14Jun2016.csv
14
1. Journey Data Extract 01Jan-05Jan13.csv
15
1. Journey Data Extract 04Jan-31Jan 12.csv
16
1. Journey Data Extract 05Jan14-02Feb14.csv
17
10. Journey Data Extract 18Aug-13Sep13.csv
18
10. Journey Data Extract 21Aug-22 Aug12.csv
19
100JourneyDataExtract07Mar2018-13Mar2018.csv
20
101JourneyDataExtract14Mar2018-20Mar2018.csv
21
102JourneyDataExtract21Mar2018-27Mar2018.csv
22
103JourneyDataExtract28Mar201